# Fase 1 · Corpus de transcripciones oficiales del BCE (Bloque 2)

**Objetivo:** construir una tabla, frase a frase, con todas las ruedas de prensa de la era Lagarde (diciembre de 2019 – septiembre de 2026): declaración inicial, preguntas de los periodistas y respuestas del panel. Es la base del clasificador de postura (fase 2), del histórico (fase 3) y del chat (fase 4).

## 1. Fuente de datos

Usamos las páginas HTML oficiales del BCE en lugar del dataset de discursos que publica el propio banco. El dataset es texto plano, mientras que el HTML conserva la estructura que necesitamos: las preguntas de los periodistas aparecen en negrita y, cuando responden presidenta y vicepresidente, cada intervención empieza por "President:" o "Vice-President:". Eso permite separar declaración, preguntas y respuestas, y atribuir cada respuesta a su hablante, sin heurísticas sobre el texto. El BCE permite reproducir sus contenidos citando la fuente.

La página índice del BCE no incluye el listado de ruedas de prensa en su HTML, así que lo leemos de los ficheros que la web carga para cada año.

In [ ]:
# autoreload aplica los cambios de src/ sin reiniciar el kernel
%load_ext autoreload
%autoreload 2

import time
import pandas as pd

from src.config import HISTORY_DIR, RAW_DIR
from src.text.corpus import listar_urls, descargar_html

## 2. Listado de ruedas de prensa

Recorremos los años 2019-2026 y nos quedamos con las ruedas de Lagarde (la primera es del 12 de diciembre de 2019). Control esperado: 8 ruedas por año, 1 en 2019 y 6 en 2026 (hasta septiembre).

In [ ]:
# Recorremos los años de la era Lagarde (2019 incluye todavía ruedas de Draghi)
urls = {}
for year in range(2019, 2027):
    urls.update(listar_urls(year))
    time.sleep(1)  # pausa de cortesía entre peticiones a la web del BCE

# Solo las ruedas de Lagarde
urls = dict(sorted((f, u) for f, u in urls.items() if f >= "2019-12-01"))

print(f"{len(urls)} ruedas de prensa")
print("Primera:", next(iter(urls.items())))
print("Última: ", list(urls.items())[-1])

In [ ]:
# Control: número de ruedas por año
pd.Series(list(urls)).str[:4].value_counts().sort_index()

In [ ]:
# Guardamos el listado para no tener que volver a consultarlo en los pasos siguientes
listado = pd.DataFrame({"date": list(urls), "url": list(urls.values())})
listado.to_csv(HISTORY_DIR / "press_conferences.csv", index=False)
print(f"Guardado: {HISTORY_DIR / 'press_conferences.csv'} ({len(listado)} filas)")

## 3. Descarga de las transcripciones

Descargamos el HTML de cada rueda de prensa y lo guardamos en `data/raw/ecb_html/` (fuera de git). Trabajar sobre una copia local tiene dos ventajas: el procesado es reproducible aunque la web del BCE cambie, y podemos ajustar el lector tantas veces como haga falta sin volver a descargar nada. Entre peticiones hacemos una pausa de un segundo para no sobrecargar el servidor.

In [ ]:
# Descarga con caché: los ficheros que ya existen no se vuelven a pedir
listado = pd.read_csv(HISTORY_DIR / "press_conferences.csv")
html_dir = RAW_DIR / "ecb_html"
rutas = descargar_html(listado, html_dir)

tamanos_kb = pd.Series({r.stem: r.stat().st_size / 1024 for r in rutas})
print(f"{len(rutas)} ficheros en {html_dir}")
print(f"Tamaño total: {tamanos_kb.sum() / 1024:.1f} MB")

In [ ]:
# Control: los ficheros más pequeños. Una página de error pesaría muy poco
# comparada con una transcripción completa
tamanos_kb.sort_values().head(5).round(1)